In [ ]:
#!pip install yfinance pandas-datareader
import pandas as pd
import yfinance as yf
import pandas_datareader.data as web
import datetime
import warnings
warnings.filterwarnings("ignore")
print("Import thư viện thành công!")

ModuleNotFoundError: No module named 'pandas_datareader'

# PHẦN 1. XÂY DỰNG DỮ LIỆU VÀ CƠ SỞ LÝ THUYẾT
## Phân tích tác động của lãi suất và lạm phát đến lợi suất cổ phiếu Microsoft (MSFT)
**Cổ phiếu:** Microsoft Corporation (MSFT)  
**Thời gian nghiên cứu:** 01/2021 – 08/2026  
**Tần suất dữ liệu:** Monthly  
### Các biến nghiên cứu
- **Y:** Monthly Return của MSFT
- **X1:** Federal Funds Effective Rate (Interest Rate)
- **X2:** Inflation Rate (CPI YoY)
### Nguồn dữ liệu
- Yahoo Finance: dữ liệu giá cổ phiếu MSFT
- FRED: Federal Funds Effective Rate (FEDFUNDS)
- FRED: Consumer Price Index (CPIAUCSL)

## 1.2. Thu thập dữ liệu
Dữ liệu được thu thập theo tần suất tháng (Monthly) trong khoảng thời gian từ tháng 01/2021 đến tháng 08/2026.
Đối với dữ liệu cổ phiếu Microsoft, nghiên cứu sử dụng giá đóng cửa điều chỉnh (Adjusted Close) từ Yahoo Finance. Adjusted Close được sử dụng làm giá đại diện để tính Monthly Return.
Đối với lãi suất, nghiên cứu sử dụng Federal Funds Effective Rate (FEDFUNDS) từ FRED.
Đối với lạm phát, nghiên cứu sử dụng chỉ số CPIAUCSL từ FRED và tính tỷ lệ lạm phát theo phương pháp Year-over-Year (YoY).

In [ ]:
# Lấy dữ liệu từ năm 2020 để có đủ dữ liệu CPI
# phục vụ tính Inflation YoY cho các tháng đầu năm 2021.
start_date = datetime.datetime(2020, 1, 1)
end_date = datetime.datetime(2026, 9, 1)
print("Start date:", start_date.date())
print("End date:", end_date.date())

In [ ]:
print("Đang tải dữ liệu MSFT từ Yahoo Finance...")
msft_df = yf.download(
    "MSFT",
    start=start_date,
    end=end_date,
    auto_adjust=False,
    progress=False
)
print("Đã tải dữ liệu MSFT.")
print("Số dòng:", len(msft_df))
display(msft_df.head())

In [ ]:
print("Các cột dữ liệu:")
print(msft_df.columns)
print("\nThông tin dữ liệu:")
msft_df.info()

In [ ]:
# Lấy Adjusted Close
msft_monthly = (
    msft_df["Adj Close"]
    .resample("ME")
    .last()
)
if isinstance(msft_monthly, pd.DataFrame):
    msft_monthly = msft_monthly["MSFT"]
msft_monthly = msft_monthly.to_frame(name="MSFT_Price")
print("Dữ liệu MSFT theo tháng:")
display(msft_monthly.head())
print("\n5 tháng cuối:")
display(msft_monthly.tail())

### Giải thích
Dữ liệu giá cổ phiếu ban đầu được cung cấp theo ngày giao dịch. Để thống nhất với dữ liệu kinh tế vĩ mô, giá cổ phiếu được chuyển sang tần suất tháng.
Giá đại diện cho mỗi tháng là Adjusted Close của ngày giao dịch cuối cùng trong tháng.
Biến `MSFT_Price` chưa phải là biến phụ thuộc cuối cùng của mô hình. Biến này sẽ được sử dụng để tính Monthly Return.

In [ ]:
print("Đang tải dữ liệu vĩ mô từ FRED...")
macro_df = web.DataReader(
    ["FEDFUNDS", "CPIAUCSL"],
    "fred",
    start_date,
    end_date
)
macro_df.columns = [
    "Interest_Rate",
    "CPI"
]
print("Đã tải dữ liệu FRED.")
display(macro_df.head())

In [ ]:
# Chuyển ngày của dữ liệu FRED về cuối tháng
macro_df.index = (
    macro_df.index
    .to_period("M")
    .to_timestamp("M")
)
macro_df.index.name = "Date"
print("Dữ liệu FRED sau khi chuẩn hóa thời gian:")
display(macro_df.head())

## 1.3. Mô tả các biến nghiên cứu
### Biến phụ thuộc – Monthly Return
Biến phụ thuộc của mô hình là Monthly Return của cổ phiếu MSFT.
Monthly Return được tính dựa trên Adjusted Close cuối mỗi tháng:
Monthly Return_t = ((P_t - P_(t-1)) / P_(t-1)) × 100
Trong đó:
- P_t là Adjusted Close của MSFT tại tháng t.
- P_(t-1) là Adjusted Close của MSFT tại tháng trước.
### Biến độc lập X1 – Interest Rate
Interest Rate được đại diện bởi Federal Funds Effective Rate (FEDFUNDS), đơn vị %.
### Biến độc lập X2 – Inflation Rate
Inflation Rate được tính từ CPI theo phương pháp Year-over-Year:
Inflation_t = ((CPI_t / CPI_(t-12)) - 1) × 100
CPI_t là CPI của tháng hiện tại và CPI_(t-12) là CPI của cùng tháng năm trước.

In [ ]:
# Tính tỷ lệ lạm phát YoY
macro_df["Inflation_Rate"] = (
    macro_df["CPI"].pct_change(12) * 100
)
print("Dữ liệu sau khi tính Inflation Rate:")
display(
    macro_df[
        ["CPI", "Inflation_Rate"]
    ].head(15)
)

In [ ]:
print("Một số giá trị Inflation Rate:")
display(
    macro_df[
        ["CPI", "Inflation_Rate"]
    ].dropna().head(10)
)
print("\n5 tháng gần nhất:")
display(
    macro_df[
        ["CPI", "Inflation_Rate"]
    ].tail()
)

In [ ]:
# Ghép dữ liệu MSFT với dữ liệu vĩ mô
df = pd.merge(
    msft_monthly,
    macro_df[
        ["Interest_Rate", "Inflation_Rate"]
    ],
    left_index=True,
    right_index=True,
    how="inner"
)
# Chỉ lấy giai đoạn nghiên cứu
df = df.loc[
    "2021-01-01":"2026-08-31"
].copy()
df.index.name = "Date"
print("Dữ liệu sau khi ghép:")
display(df.head())
print("\n5 dòng cuối:")
display(df.tail())

In [ ]:
print("Kích thước dữ liệu:", df.shape)
print("\nTên các cột:")
print(df.columns.tolist())
print("\nKhoảng thời gian:")
print("Từ:", df.index.min())
print("Đến:", df.index.max())

## 1.4. Tiền xử lý dữ liệu
Sau khi ghép dữ liệu, tiến hành kiểm tra các giá trị bị thiếu, dữ liệu trùng lặp và kiểu dữ liệu.
Dữ liệu được lọc trong khoảng thời gian nghiên cứu từ tháng 01/2021 đến tháng 08/2026.
Các giá trị thiếu được kiểm tra trước khi xử lý. Nếu không có giá trị thiếu thì giữ nguyên dữ liệu. Nếu xuất hiện giá trị thiếu, các quan sát không đầy đủ sẽ được loại bỏ để tránh đưa dữ liệu không xác định vào mô hình.

In [ ]:
print("=== KIỂM TRA MISSING VALUES ===")
missing_values = df.isnull().sum()
display(missing_values)

In [ ]:
print("=== KIỂM TRA DỮ LIỆU TRÙNG LẶP ===")
duplicate_dates = df.index.duplicated().sum()
print("Số ngày bị trùng:", duplicate_dates)

In [ ]:
# Nếu có missing values thì loại bỏ các dòng bị thiếu
df = df.dropna().copy()
print("Missing values sau khi xử lý:")
print(df.isnull().sum())

In [ ]:
print("=== THÔNG TIN DATAFRAME ===")
df.info()

In [ ]:
# Làm tròn 4 chữ số thập phân để dữ liệu dễ quan sát
df = df.round(4)
display(df.head(10))

In [ ]:
print("=== THỐNG KÊ CƠ BẢN ===")
display(df.describe())

In [ ]:
print("=== DATASET CUỐI CÙNG ===")
print("Số dòng:", df.shape[0])
print("Số cột:", df.shape[1])
print("\nKhoảng thời gian:")
print(df.index.min(), "đến", df.index.max())
print("\nCác cột:")
print(df.columns.tolist())
display(df.head())
display(df.tail())

In [ ]:
# Lưu dataset vào thư mục data
output_path = "../data/msft_macro_2021_2026.csv"
df.to_csv(output_path)
print("Đã lưu dữ liệu tại:")
print(output_path)

## 1.5. Cơ sở lý thuyết
### 1.5.1. Tác động của lãi suất
Lãi suất có thể ảnh hưởng đến giá cổ phiếu thông qua tỷ lệ chiết khấu và chi phí vốn. Khi lãi suất tăng, giá trị hiện tại của các dòng tiền kỳ vọng trong tương lai có thể giảm. Bên cạnh đó, chi phí vốn tăng có thể ảnh hưởng đến hoạt động đầu tư và chi tiêu của doanh nghiệp.
Đối với các công ty công nghệ như Microsoft, sự thay đổi của lãi suất có thể ảnh hưởng đến kỳ vọng của nhà đầu tư đối với tăng trưởng và lợi nhuận trong tương lai. Tuy nhiên, chiều hướng và mức độ tác động thực tế cần được kiểm định bằng dữ liệu.
### 1.5.2. Tác động của lạm phát
Lạm phát có thể làm tăng một số chi phí hoạt động của doanh nghiệp như chi phí nhân sự, năng lượng và cơ sở hạ tầng. Nếu chi phí tăng nhanh hơn doanh thu, lợi nhuận của doanh nghiệp có thể chịu ảnh hưởng.
Ngoài ra, lạm phát còn có thể tác động gián tiếp đến thị trường chứng khoán thông qua chính sách tiền tệ và lãi suất. Vì vậy, lạm phát có thể có mối quan hệ với lợi suất cổ phiếu Microsoft và cần được kiểm định bằng mô hình hồi quy.
### 1.5.3. Mô hình nghiên cứu
Nghiên cứu sử dụng mô hình hồi quy tuyến tính đa biến:
**Monthly Return_t = β0 + β1 Interest Rate_t + β2 Inflation Rate_t + ε_t**
Trong đó:
- **Monthly Return:** lợi suất cổ phiếu MSFT theo tháng.
- **Interest Rate:** Federal Funds Effective Rate.
- **Inflation Rate:** tỷ lệ lạm phát CPI YoY.
- **β0:** hệ số chặn.
- **β1, β2:** hệ số hồi quy.
- **ε:** sai số của mô hình.
Nghiên cứu sẽ sử dụng kết quả hồi quy để đánh giá chiều hướng, mức độ và ý nghĩa thống kê của mối quan hệ giữa các biến.

## Kết luận phần 1
Dữ liệu MSFT, lãi suất và lạm phát đã được thu thập, chuẩn hóa và kiểm tra trong giai đoạn 01/2021–08/2026.
Bộ dữ liệu sau xử lý được lưu thành file CSV và sẽ được sử dụng cho các bước phân tích tiếp theo, bao gồm tính Monthly Return, phân tích dữ liệu và xây dựng mô hình hồi quy tuyến tính đa biến.